# Homework 2

## Instructions

For parts A and B, fill in the code and answers in the notebook, then save your changes.

For part C, create the necessary Python script.

Finally, upload the two files to the corresponding assignment in ILIAS.

## Part A

### Exercise A1: VAE VFL (7 points)

Implement VFL for a generative scenario with VAE-based models.
Use the same dataset (`heart`), client count (4), feature split (uniform), bottom/top model architectures, and preprocessing as in the lab.
Test the final generation quality using the lab's auxiliary classification model.
Each client have the first two encoder layers, and the last two decoder layers for the corresponding features.
The server has the last two encoder layers and the first two decoder layers for all features.

Score breakdown:
- bottom model _(1 point)_
- top model _(1 point)_
- client class _(2 points)_
- server class _(2 point)_
- test accuracy _(1 point)_

In [ ]:
from lab_3_vfl.vfl_complete import *

In [ ]:
class BottomModelVae(nn.Module):
    def __init__(self, input_dim: int, hidden_dim=6, latent_dim=4):

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        ...

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        ...

    ...

In [ ]:
class TopModelVae(nn.Module):
    def __init__(self, input_dim: int, hidden_dim=8, latent_dim=4):

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        ...

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        ...

    ...

In [ ]:
class VflClientVae:
    def __init__(
            self,  input_dim: int,
            client_data: torch.Tensor,
            lr: float, seed: int) -> None:
        ...

    def forward_encode(self, inds: torch.Tensor) -> torch.Tensor:
        ...

    def decode_and_calc_mse_grad(
            self, inds: torch.Tensor, h_dec: torch.Tensor) -> torch.Tensor:
        ...

    def backward_encode_and_step(self, split_grad: torch.Tensor) -> None:
        ...

In [ ]:
class VflServerVae:
    def __init__(
            self, clients: list[VflClientVae],
            train_set_size: int,
            lr: float, batch_size: int, seed: int) -> None:
        ...

    def run(self, nr_epochs: int) -> None:
        ...

    def sample(self, nr_samples: int, seed=0) -> torch.Tensor:
        ...

In [ ]:
lr = 0.001
seed = 0
train = torch.cat((X_train, y_train.to(X_train.dtype).unsqueeze(dim=1)), dim=1)
train_chunks = train.chunk(4, dim=1)
vfl_clients = [
    VflClientVae(train_chunk.size(1), train_chunk, lr, seed)
    for train_chunk in train_chunks
]
vfl_server = VflServerVae(vfl_clients, len(X_train), lr, 64, seed)
vfl_server.run(100)
synth = vfl_server.sample(X_train.size(0))
classifier = train_classifier(synth[:, :-1], synth[:, -1].to(y_train.dtype))
test_acc = test_classifier(classifier, X_test, y_test)
f"Test accuracy of classifier trained on VFL VAE data: {test_acc:.2f}"

### Exercise B1: MD-GANs (5 points)

Implement multi-discriminator federated GANs.
Use the same dataset, client count, generator/discriminator model architectures, and preprocessing as in the lab.
In the multi-discriminator GAN setup, discriminator clients receive at every step fake samples from the generator server.
Clients first optimize their discriminator using a batch of real samples and one using the fake samples from the server.
They then compute and relay the gradients for the samples received from the server.
The server averages all client gradients, then optimizes its generator.
Consider the setting in which all clients participate at each step, and their gradients get averaged without weighting.

Score breakdown:
- discriminator client class _(2 points)_
- generator server class _(2 point)_
- integration _(1 point)_

In [ ]:
from typing import cast

from lab_4_fed_gen_ai.gan_complete import *

In [ ]:
class DiscriminatorClient:
    def __init__(
            self, client_data: datasets.Dataset,
            lr: float, batch_size: int,
            beta1: float, seed: int) -> None:
        ...

    def get_real_batch(self) -> torch.Tensor:
        ...

    # output is (gradients, generator loss, discriminator loss)
    # the losses are to allow for monitoring, not correctness
    def update(self, fake_data: torch.Tensor) -> tuple[torch.Tensor, float, float]:
        ...

In [ ]:
class GeneratorServer:
    def __init__(
            self, clients: list[DiscriminatorClient],
            lr: float, batch_size: int,
            beta1: float, seed: int) -> None:
        ...

    def run(self, nr_steps: int) -> None:
        ...

    def sample(self, nr_samples: int, sample_seed=0) -> torch.Tensor:
        ...

In [ ]:
clients = [
    DiscriminatorClient(subset, 2e-4, 16, 0.5, 0)
    for subset in CLIENT_SPLIT]
server = GeneratorServer(
    clients=clients,
    lr=2e-4,
    batch_size=16,
    beta1=0.5,
    seed=0,
)
server.run(nr_steps=1_000)
_ = plot(server.sample(16))

### Exercise C1: Horizontal Decentralized RL (5 points)

Implement a decentralized GRPO for a horizontal setting.
Unlike in the vertical case, where each worker had a different set of prompts and provided all completions for each, in the horizontal case, all workers have the same set of prompts at each step, and completions for each get split across all clients.
Use the same dataset, worker count, base model, and preprocessing as in the lab.
As with the vertical decentralized RL example from the lab, your Python script should handle spinning up multiple workers and define their shared logic.